# Real-data validation, Part 3: real ambient urban PM2.5 (Tier 4)

Reproduces the code behind paper Section 3.6 of "Validating a Machine-Learning-Accelerated AIOMFAC Pipeline
for Secondary Organic Aerosol Viscosity": testing the ML-Tg + AIOMFAC + VTF viscosity pipeline against
Ullah, A., Lee, J. Y., Wu, Z., Jang, K.-S., and Song, M. (2026), "Highly viscous phase behavior of
organic-rich urban PM2.5", *Atmos. Chem. Phys.*, 26, 7311-7324 -- RH-resolved viscosity of real,
field-collected Seoul/Beijing PM2.5 filter extracts (poke-and-flow + fluid-dynamics simulation), reporting
mean viscosities of ~1e4-1e8 Pa s across RH 25-45% at an organic:inorganic mass ratio (OIR) of 2:1-8:1 and
bulk O:C = 0.45-0.52.

Unlike the pure-compound and chamber-SOA tiers (which are ion-free water-organic systems), real ambient PM2.5
also contains a substantial inorganic (sulfate, mostly as ammonium sulfate) fraction, so this uses the
organic-inorganic viscosity mixing extension already implemented in this project's `aiomfac_py.viscosity`
module -- two of Lilek and Zuend (2022)'s three mixing rules, "aquelec" and "aquorg" (their Sect. 3.4.1-3.4.2)
-- rather than the ion-free `organic_mixture_viscosity` used in the pure-compound/chamber-SOA tiers. The
paper reports only bulk organic mass (OM), not molecular composition, so as in the chamber-SOA tier we use
the field's standard alpha-pinene-SOA proxy pair (pinonic acid, O:C=0.30; pinic acid, O:C=0.44), whose O:C
brackets the paper's measured bulk O:C (0.45-0.52) on the oxidized side.

## 0. Setup

In [1]:
import os
if not os.path.exists("aiomfac-python"):
    !git clone -q -b main https://github.com/SeoulTechPSE/aiomfac-python.git
%cd aiomfac-python
!python setup_aiomfac.py --smiles --no-editable -q 2>&1 | tail -20
!pip install -q scipy matplotlib numpy
%cd ..
print("setup done")

/content/aiomfac-python
Editable : False
Clean    : False

Installing aiomfac_py with extras [carbonate,smiles] ...
$ /usr/bin/python3 -m pip install -q .[carbonate,smiles]

Verifying installation (in this process, not a subprocess)...
aiomfac_py 0.0.24  (validated against AIOMFAC-web commit b9cb96d0eb22)
  scipy 1.16.3 OK
  epam.indigo OK, version 1.46.0.0-gb36d38f52-x86_64-linux-gnu-11.2.1

aiomfac_py setup complete!

Next steps:
  from aiomfac_py import read_input_file, ActivityModel
  See notebooks/01_quickstart.ipynb for worked examples.
/content
setup done


**Note:** this validation uses `aiomfac_py.viscosity.aquelec_viscosity` / `aquorg_viscosity`, which were
already present in the upstream repo before this validation was written, so it does not depend on the
same-session LLE/dissociation fixes used by the companion LLPS-paper notebook.

In [2]:
import sys
sys.path.insert(0, "aiomfac-python/src")
import json
import numpy as np
from aiomfac_py.s2as import smiles_to_components
from aiomfac_py import ActivityModel, Component
from aiomfac_py.viscosity import pure_organic_viscosity_vtf, aquelec_viscosity, aquorg_viscosity

T_K = 290.0  # paper's poke-and-flow cell temperature

WATER = Component(1, "Water", ((16, 1),))
AS = Component(3, "(NH4)2SO4", ((204, 2), (261, 1)))

PROXIES = {
    "pinonic_acid": {"smiles": "CC(=O)C1CC(C1(C)C)CC(=O)O", "o_to_c": 0.30, "tg_measured_K": 265.0},
    "pinic_acid": {"smiles": "CC1(C(CC1C(=O)O)CC(=O)O)C", "o_to_c": 0.4444, "tg_measured_K": 268.1},
}

# Paper's reported OIR range across its 6 filter samples (Table 2): 2:1 to 8:1.
OIR_GRID = [2.0, 3.0, 8.0]
RH_GRID = np.array([0.45, 0.40, 0.35, 0.30, 0.25, 0.20])

# Paper's own text constraint (Sect 3.3): mean viscosities ~1e4-1e8 Pa s across RH 25-45%.
PAPER_LOG10_ETA_LO, PAPER_LOG10_ETA_HI = 4.0, 8.0


def find_wtf_water_for_aw(model, target_aw, T_K, f_org, lo=0.0005, hi=0.9995, tol=1e-3, max_iter=60):
    """Bisection on wtf_water (ternary water/organic/AS composition fixed at organic mass fraction f_org
    of the non-water mass)."""
    def aw_at(wtf_w):
        wtf_org = (1.0 - wtf_w) * f_org
        wtf_as = (1.0 - wtf_w) * (1.0 - f_org)
        res = model.evaluate([wtf_w, wtf_org, wtf_as], T_K, basis="mass")
        return float(res.activity[0]), res

    aw_lo, _ = aw_at(lo)
    aw_hi, _ = aw_at(hi)
    if not (aw_lo <= target_aw <= aw_hi):
        return None, None
    res = None
    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        aw_mid, res = aw_at(mid)
        if abs(aw_mid - target_aw) < tol:
            return mid, res
        if aw_mid < target_aw:
            lo = mid
        else:
            hi = mid
    return mid, res


print(f"T={T_K} K; organic proxies bracket paper's measured bulk O:C=0.45-0.52:")
for name, info in PROXIES.items():
    print(f"  {name}: O:C={info['o_to_c']}")

results = {}
for name, info in PROXIES.items():
    r = smiles_to_components([info["smiles"]], water_as_component1=True)
    organic = Component(2, name, r.components[1].subgroups)
    model = ActivityModel([WATER, organic, AS])
    eta0 = pure_organic_viscosity_vtf(T_K, info["tg_measured_K"])

    print(f"\n=== {name} (O:C={info['o_to_c']}, pure-component eta0={eta0:.2e} Pa s) + AS, vs. Ullah et al. "
          f"(2026) ambient PM2.5 ===")
    for oir in OIR_GRID:
        f_org = oir / (oir + 1.0)
        print(f"  OIR={oir:.0f}:1 (organic mass fraction of dry matter={f_org:.3f})")
        for rh in RH_GRID:
            wtf_w, res = find_wtf_water_for_aw(model, rh, T_K, f_org)
            if wtf_w is None:
                print(f"    RH={rh*100:.0f}%  (no solution in composition range)")
                continue
            v_elec = aquelec_viscosity(model, res, T_K, {2: eta0})
            v_org = aquorg_viscosity(model, res, T_K, {2: eta0})
            log_elec = np.log10(v_elec.eta_pas) if v_elec.eta_pas > 0 else np.nan
            log_org = np.log10(v_org.eta_pas) if v_org.eta_pas > 0 else np.nan
            in_band = "" if 0.20 <= rh <= 0.45 else " (outside paper's 25-45% band)"
            flag_e = "OK" if PAPER_LOG10_ETA_LO <= log_elec <= PAPER_LOG10_ETA_HI else "--"
            flag_o = "OK" if PAPER_LOG10_ETA_LO <= log_org <= PAPER_LOG10_ETA_HI else "--"
            print(f"    RH={rh*100:.0f}%  aquelec log10(eta)={log_elec:+.2f} [{flag_e}]   "
                  f"aquorg log10(eta)={log_org:+.2f} [{flag_o}]{in_band}")
            results.setdefault(name, []).append(dict(oir=oir, rh=float(rh), log10_eta_aquelec=float(log_elec),
                                                        log10_eta_aquorg=float(log_org)))

print(f"\npaper\'s own text constraint: log10(eta) in [{PAPER_LOG10_ETA_LO:.0f}, {PAPER_LOG10_ETA_HI:.0f}] "
      f"(~1e4-1e8 Pa s) across RH 25-45%, T~290K, exceeding/matching sucrose-AS and exceeding citric acid-AS "
      f"lab proxies at comparable RH (paper Fig. 5).")

with open("ambient_pm25_validation_results.json", "w") as f:
    json.dump(dict(T_K=T_K, oir_grid=OIR_GRID, rh_grid=RH_GRID.tolist(),
                    paper_band_log10_eta=[PAPER_LOG10_ETA_LO, PAPER_LOG10_ETA_HI], results=results), f, indent=2)
print("\nwrote ambient_pm25_validation_results.json")

T=290.0 K; organic proxies bracket paper's measured bulk O:C=0.45-0.52:
  pinonic_acid: O:C=0.3
  pinic_acid: O:C=0.4444

=== pinonic_acid (O:C=0.3, pure-component eta0=4.10e+06 Pa s) + AS, vs. Ullah et al. (2026) ambient PM2.5 ===
  OIR=2:1 (organic mass fraction of dry matter=0.667)
    RH=45%  aquelec log10(eta)=+0.29 [--]   aquorg log10(eta)=-0.68 [--]
    RH=40%  aquelec log10(eta)=+0.74 [--]   aquorg log10(eta)=-0.44 [--]
    RH=35%  aquelec log10(eta)=+1.24 [--]   aquorg log10(eta)=-0.20 [--]
    RH=30%  aquelec log10(eta)=+1.81 [--]   aquorg log10(eta)=+0.06 [--]
    RH=25%  aquelec log10(eta)=+2.44 [--]   aquorg log10(eta)=+0.32 [--]
    RH=20%  aquelec log10(eta)=+3.10 [--]   aquorg log10(eta)=+0.58 [--]
  OIR=3:1 (organic mass fraction of dry matter=0.750)
    RH=45%  aquelec log10(eta)=+1.44 [--]   aquorg log10(eta)=+0.11 [--]
    RH=40%  aquelec log10(eta)=+1.93 [--]   aquorg log10(eta)=+0.37 [--]
    RH=35%  aquelec log10(eta)=+2.44 [--]   aquorg log10(eta)=+0.63 [--]
   

## Summary: mean offset from the paper's band, by OIR

In [3]:
lo, hi = PAPER_LOG10_ETA_LO, PAPER_LOG10_ETA_HI
mid = (lo + hi) / 2
for proxy, rows in results.items():
    print(f"\n{proxy}:")
    for oir in sorted(set(r["oir"] for r in rows)):
        sub = [r for r in rows if r["oir"] == oir]
        offs_e = [mid - r["log10_eta_aquelec"] for r in sub]
        offs_o = [mid - r["log10_eta_aquorg"] for r in sub]
        in_band_e = sum(1 for r in sub if lo <= r["log10_eta_aquelec"] <= hi)
        print(f"  OIR={oir}: aquelec mean offset from band-mid={sum(offs_e)/len(offs_e):+.2f} log10 units, "
              f"in-band count={in_band_e}/{len(sub)}; aquorg mean offset={sum(offs_o)/len(offs_o):+.2f}")
print("\n(Table 3 in the paper draft reports the pinic-acid rows of this summary.)")


pinonic_acid:
  OIR=2.0: aquelec mean offset from band-mid=+4.40 log10 units, in-band count=0/6; aquorg mean offset=+6.06
  OIR=3.0: aquelec mean offset from band-mid=+3.27 log10 units, in-band count=1/6; aquorg mean offset=+5.26
  OIR=8.0: aquelec mean offset from band-mid=+1.93 log10 units, in-band count=3/6; aquorg mean offset=+3.56

pinic_acid:
  OIR=2.0: aquelec mean offset from band-mid=+4.37 log10 units, in-band count=0/6; aquorg mean offset=+6.05
  OIR=3.0: aquelec mean offset from band-mid=+3.50 log10 units, in-band count=1/6; aquorg mean offset=+5.36
  OIR=8.0: aquelec mean offset from band-mid=+2.43 log10 units, in-band count=2/6; aquorg mean offset=+3.87

(Table 3 in the paper draft reports the pinic-acid rows of this summary.)


## Interpretation

Both mixing rules systematically **underpredict** the measured ambient viscosity band across most of the
tested OIR/RH range, closing in only at the driest, most organic-rich corner (OIR=8:1, RH<=30%). `aquelec`
is consistently closer to the measured band than `aquorg`. This is read as a calibrating negative result,
directionally consistent with (a) the glassy-regime weakness of the underlying G2020 mixing model and (b) a
single small-molecule proxy under-representing the viscosity-raising complexity of real, aged ambient OA --
not evidence of a problem specific to the organic-inorganic (aquelec/aquorg) extension itself, since the
inorganic-free chamber-SOA comparisons show qualitatively the same direction and rough magnitude of error.